# Session 17: AutoML and Transparent Models

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL17)  
**Level**: 🔴 Advanced

---

## 📋 Objectives
1. Deploy Artificial Intelligence to write its own ML algorithms (AutoML via TPOT).
2. Understand the danger of Black-Box algorithms.
3. Implement SHAP graphs to force Black-Box models to mathematically explain themselves.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target  # 0 = Malignant, 1 = Benign

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("✅ Breast cancer data loaded.")

---
## §1 — AutoML (Genetic Algorithms)
Instead of guessing if we should use `StandardScaler` or `MinMaxScaler`, and guessing between `RandomForest` and `LogisticRegression`, let's just make the AI test every combination mathematically against each other!

In [ ]:
# !pip install tpot
from tpot import TPOTClassifier

# Set generations=3 so this runs in 60 seconds. 
# In the real world, you might set it to 100 and let it run all night!
tpot = TPOTClassifier(generations=3, population_size=10, verbosity=2, random_state=42)

# Begin the automated pipeline search!
tpot.fit(X_train, y_train)

print(f"\nTPOT Final Champion Pipeline Accuracy: {tpot.score(X_test, y_test) * 100:.2f}%")

In [ ]:
# The greatest feature of TPOT is that it translates the winning model back into Python code so you can copy/paste it into your Github repository!
tpot.export('tpot_champion_pipeline.py')
print("Exported the winning code to 'tpot_champion_pipeline.py'!")

---
## §2 — The Black-Box Transparency Problem (SHAP)
AutoML is amazing. But what if `TPOT` decides the best mathematical model is an incredibly complex `ExtraTreesClassifier` chained with a polynomial feature generator? 
That is a **Black-Box Model**. If the AI tells a doctor a patient has cancer, the doctor will ask *"Why?"*
If you say *"I don't know, the AI black-box just said so"*, you will be fired.

We use **SHAP values** to fix this.

In [ ]:
# !pip install shap
import shap
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Train a very complex Black-box (100 independent decision trees voting together)
black_box_model = RandomForestClassifier(n_estimators=100, random_state=42)
black_box_model.fit(X_train, y_train)

# 2. Deploy SHAP to analyze the math inside the Black Box
explainer = shap.TreeExplainer(black_box_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Let's look at the GLOBAL logic. Which features does the AI care about overall?

# Depending on your SHAP version, shap_values might be a list for multi-class.
# [1] represents explaining the logic behind predicting the "Benign" class.
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], X_test)
else:
    shap.summary_plot(shap_values, X_test)

### Read the Chart!
- Notice the `worst perimeter` and `worst concave points` are at the top.
- Red dots mean the numeric value of that feature was HIGH.
- Look at `worst perimeter`. The red dots are all on the LEFT side of the zero line (Negative SHAP value).
- **Executive Conclusion:** "When the perimeter of the tumor is massively high (red dots), it subtracts from the probability of the tumor being Benign. Mathematically, large perimeters drive Malignant predictions!"


**You have successfully cracked open the Black Box and achieved MLOps Transparency!**